Phase 03 : Master Data 

In [1]:
import pandas as pd
import hashlib

df = pd.read_csv(
    "online_retail_clean.csv",
    dtype={"transaction_id": str, "product_id": str, "customer_id": str},
    parse_dates=["event_time"],
)
print(len(df))

397884


In [2]:
def stable(text, n):
    h = hashlib.md5(text.encode("utf-8")).hexdigest()
    return int(h[:8], 16) % n

GENDER     = ["Male", "Female"]
AGE        = ["0-17", "18-25", "26-35", "36-45", "46-50", "51-55", "55+"]
OCCUPATION = ["Student", "Engineer", "Teacher", "Executive", "Clerk",
              "Healthcare", "Retail", "Homemaker", "Artist", "Other"]
CITY       = ["A", "B", "C"]
STAY       = ["0", "1", "2", "3", "4+"]
MARITAL    = ["Single", "Married"]
CATEGORY   = ["Home Decor", "Kitchen", "Stationery", "Toys", "Seasonal Gifts",
              "Bags", "Jewelry", "Garden", "Party", "Other"]
SUPPLIERS  = [("SUP01", "Northwind Supplies"), ("SUP02", "Atlantic Gifts Ltd"),
              ("SUP03", "Midlands Wholesale"), ("SUP04", "Celtic Craft Co"),
              ("SUP05", "EuroPack Partners")]

In [3]:
cust = df.sort_values("event_time", kind="stable").drop_duplicates("customer_id")
cust = cust[["customer_id", "country"]]

cust["gender"] = cust["customer_id"].apply(lambda c: GENDER[stable(c, 2)])

cust["age_group"] = cust["customer_id"].apply(lambda c: AGE[stable(c + "age", 7)])

cust["occupation"] = cust["customer_id"].apply(lambda c: OCCUPATION[stable(c + "occ", 10)])

cust["city_category"] = (cust["customer_id"] + cust["country"]).apply(lambda t: CITY[stable(t, 3)])

cust["stay_in_current_city_years"] = cust["customer_id"].apply(lambda c: STAY[stable(c + "stay", 5)])

cust["marital_status"] = cust["customer_id"].apply(lambda c: MARITAL[stable(c + "mar", 2)])

cust = cust.drop(columns="country").sort_values("customer_id").reset_index(drop=True)
print(len(cust), "customers")
cust.head()

4338 customers


,customer_id,gender,age_group,occupation,city_category,stay_in_current_city_years,marital_status
0,12346,Female,26-35,Clerk,C,0,Married
1,12347,Female,36-45,Clerk,A,3,Married
2,12348,Male,0-17,Other,C,3,Single
3,12349,Female,55+,Executive,B,4+,Married
4,12350,Female,46-50,Healthcare,A,2,Married


In [4]:
p = df[["product_id", "product_name"]].copy()
p["product_name"] = p["product_name"].fillna("").str.strip()
p["name_len"] = p["product_name"].str.len()

# longest description per product (ties: first in file order)
prod = p.sort_values("name_len", ascending=False, kind="stable").drop_duplicates("product_id")
prod = prod[["product_id", "product_name"]]
prod.loc[prod["product_name"] == "", "product_name"] = "UNKNOWN"

prod["product_category"] = prod["product_id"].apply(lambda x: CATEGORY[stable(x + "cat", 10)])
prod["supplier_id"]      = prod["product_id"].apply(lambda x: SUPPLIERS[stable(x, 5)][0])
prod["supplier_name"]    = prod["product_id"].apply(lambda x: SUPPLIERS[stable(x, 5)][1])

prod = prod.sort_values("product_id").reset_index(drop=True)
print(len(prod), "products")
prod.head()

3665 products


,product_id,product_name,product_category,supplier_id,supplier_name
0,10002,INFLATABLE POLITICAL GLOBE,Home Decor,SUP04,Celtic Craft Co
1,10080,GROOVY CACTUS INFLATABLE,Home Decor,SUP03,Midlands Wholesale
2,10120,DOGGY RUBBER,Jewelry,SUP02,Atlantic Gifts Ltd
3,10123C,HEARTS WRAPPING TAPE,Bags,SUP01,Northwind Supplies
4,10124A,SPOTS ON RED BOOKCOVER TAPE,Party,SUP05,EuroPack Partners


In [5]:
print("Customers with more than one country:",
      (df.groupby("customer_id")["country"].nunique() > 1).sum())
print("Products named UNKNOWN:", (prod["product_name"] == "UNKNOWN").sum())
print("Duplicate customer ids:", cust["customer_id"].duplicated().sum())
print("Duplicate product ids:", prod["product_id"].duplicated().sum())

print("\nCustomer 17850:")
print(cust[cust["customer_id"] == "17850"].T)

Customers with more than one country: 8
Products named UNKNOWN: 0
Duplicate customer ids: 0
Duplicate product ids: 0

Customer 17850:
                                4016
customer_id                    17850
gender                        Female
age_group                      51-55
occupation                  Engineer
city_category                      A
stay_in_current_city_years         1
marital_status               Married


In [6]:
cust.to_csv("customer_master_data.csv", index=False)
prod.to_csv("product_master_data.csv", index=False)
print("Saved:", len(cust), "customers,", len(prod), "products")

Saved: 4338 customers, 3665 products
